In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("visa_dataset_training_ready.csv")

X = df.drop("processing_days", axis=1)
y = df["processing_days"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Results")
print("MAE:", mae_lr)
print("RMSE:", rmse_lr)
print("R2 Score:", r2_lr)

Linear Regression Results
MAE: 7.618811450211134
RMSE: 23.42918426982933
R2 Score: 0.0006102433645391869


In [2]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=300,max_depth=20,random_state=42,n_jobs=-1)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("\nRandom Forest Results")
print("MAE:", mae_rf)
print("RMSE:", rmse_rf)
print("R2 Score:", r2_rf)


Random Forest Results
MAE: 7.551577313542078
RMSE: 23.277272048765017
R2 Score: 0.013528091931172548


In [3]:
!pip install xgboost

In [4]:
from xgboost import XGBRegressor

xgb = XGBRegressor(n_estimators=400,learning_rate=0.05,max_depth=6,subsample=0.8,colsample_bytree=0.8,random_state=42)

xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("\nXGBoost Results")
print("MAE:", mae_xgb)
print("RMSE:", rmse_xgb)
print("R2 Score:", r2_xgb)


XGBoost Results
MAE: 7.596213340759277
RMSE: 22.997793340561095
R2 Score: 0.03707420825958252


In [6]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "XGBoost"],
    "MAE": [mae_lr, mae_rf, mae_xgb],
    "RMSE": [rmse_lr, rmse_rf, rmse_xgb],
    "R2 Score": [r2_lr, r2_rf, r2_xgb]
})

print(results)

               Model       MAE       RMSE  R2 Score
0  Linear Regression  7.618811  23.429184  0.000610
1      Random Forest  7.551577  23.277272  0.013528
2            XGBoost  7.596213  22.997793  0.037074


**XGBoost without fine-tuning**

In [7]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

xgb_baseline = XGBRegressor(n_estimators=400,learning_rate=0.05,max_depth=6,random_state=42)

xgb_baseline.fit(X_train, y_train)

y_pred_base = xgb_baseline.predict(X_test)

mae_base = mean_absolute_error(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_base = r2_score(y_test, y_pred_base)

print("Baseline XGBoost")
print("MAE:", mae_base)
print("RMSE:", rmse_base)
print("R2:", r2_base)

Baseline XGBoost
MAE: 7.597809314727783
RMSE: 23.12004539406092
R2: 0.0268094539642334


In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [300, 400, 500],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1],
    'colsample_bytree': [0.7, 0.8, 1]
}

xgb_model = XGBRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

best_xgb = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)

Fitting 3 folds for each of 243 candidates, totalling 729 fits
Best Parameters: {'colsample_bytree': 1, 'learning_rate': 0.01, 'max_depth': 8, 'n_estimators': 300, 'subsample': 0.7}


In [12]:
y_pred_tuned = best_xgb.predict(X_test)

mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2_tuned = r2_score(y_test, y_pred_tuned)

print("\nFine-Tuned XGBoost")
print("MAE:", mae_tuned)
print("RMSE:", rmse_tuned)
print("R2:", r2_tuned)


Fine-Tuned XGBoost
MAE: 7.449281215667725
RMSE: 23.035328145555848
R2: 0.03392839431762695


In [13]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["XGBoost (Baseline)", "XGBoost (Fine-Tuned)"],
    "MAE": [mae_base, mae_tuned],
    "RMSE": [rmse_base, rmse_tuned],
    "R2 Score": [r2_base, r2_tuned]
})

print(results)

                  Model       MAE       RMSE  R2 Score
0    XGBoost (Baseline)  7.597809  23.120045  0.026809
1  XGBoost (Fine-Tuned)  7.449281  23.035328  0.033928
